In [3]:
import numpy as np
from pathlib import Path

Checking shape of all the files 

In [4]:

raw = Path("../../data/raw")
for f in sorted(raw.glob("*.txt")):
    try:
        a = np.loadtxt(f)
        print(f"{f.name:18s} shape={str(a.shape):14s} "
              f"NaN={np.isnan(a).sum():<4d} min={a.min():9.3f} max={a.max():9.3f}")
    except Exception as e:
        print(f"{f.name:18s} FAILED: {e}")

H12.txt            shape=(32, 50000)    NaN=0    min=-3596.500 max=  655.190
H12_w3.txt         shape=(49,)          NaN=0    min=    5.030 max=    8.215
H3.txt             shape=(32, 50000)    NaN=0    min= -894.310 max= 1017.800
H3_w3.txt          shape=(49,)          NaN=0    min=    3.246 max=    5.013
H4.txt             shape=(32, 50000)    NaN=0    min= -533.490 max=  823.180
H4_w3.txt          shape=(49,)          NaN=0    min=    2.989 max=    6.362
H5.txt             shape=(32, 50000)    NaN=0    min=-1258.500 max= 1621.800
H5_w3.txt          shape=(49,)          NaN=0    min=    8.467 max=   17.003
H6.txt             shape=(32, 50000)    NaN=0    min= -786.350 max= 1933.300
H6_w3.txt          shape=(49,)          NaN=0    min=    3.478 max=    7.175
H7.txt             shape=(32, 50000)    NaN=0    min= -626.380 max= 1142.100
H7_w3.txt          shape=(49,)          NaN=0    min=   11.333 max=   19.944
Hurst_H12.txt      shape=(17000, 2)     NaN=0    min=    0.767 max=    0.990

Checking which columns in the Hurst value files for a given patient are F3 by matching the hurst values given in T2 of the original study using mean and standard deviation of each column

In [5]:
#hurst coeffecients for each participant (the file names don't exactly match the names in the table, for example 
# Hurst_H3.txt corresponds to H1 in the table)  

table2 = {"H3": 0.9505, "H4": 0.9436, "H5": 0.9522, "H6": 0.9807, "H7": 0.9640, "H12": 0.9520,
          "P1": 0.9357, "P2": 0.9170, "P8": 0.9103, "P9": 0.9118, "P10": 0.9591, "P11": 0.9064}

#for each patient, finds their hurst coeffecients file and takes the mean of all the columns, then sees wich column
#is closest to the patient's mean hurst from the table in the paper. For all the files with 32 columns, we get col 2 
#corresponding to the F3 hurst value. For the files with 2 columns, we get col 0 corresponding the Hurst value.
for pid, target in table2.items():
    f = next(raw.glob(f"Hurst_{pid}.txt"), None) or next(raw.glob(f"Hurst_{pid.lower()}.txt"))
    means = np.loadtxt(f).mean(axis=0)
    best = np.argmin(np.abs(means - target))
    print(f"{pid:4s} best col={best:2d}  mean={means[best]:.4f}  table2={target}  diff={means[best]-target:+.4f}")

H3   best col= 2  mean=0.9505  table2=0.9505  diff=+0.0000
H4   best col= 2  mean=0.9436  table2=0.9436  diff=-0.0000
H5   best col= 2  mean=0.9522  table2=0.9522  diff=+0.0000
H6   best col= 2  mean=0.9807  table2=0.9807  diff=-0.0000
H7   best col= 2  mean=0.9640  table2=0.964  diff=-0.0000
H12  best col= 0  mean=0.9520  table2=0.952  diff=-0.0000
P1   best col= 2  mean=0.9357  table2=0.9357  diff=-0.0000
P2   best col= 2  mean=0.9170  table2=0.917  diff=+0.0000
P8   best col= 2  mean=0.9103  table2=0.9103  diff=+0.0000
P9   best col= 2  mean=0.9118  table2=0.9118  diff=-0.0000
P10  best col= 2  mean=0.9591  table2=0.9591  diff=-0.0000
P11  best col= 0  mean=0.9064  table2=0.9064  diff=-0.0000


In [6]:
table2_sd = {"H3": 0.0041, "H4": 0.0115, "H5": 0.0159, "H6": 0.0076, "H7": 0.0038, "H12": 0.0061,
             "P1": 0.0069, "P2": 0.0094, "P8": 0.0079, "P9": 0.0084, "P10": 0.0262, "P11": 0.0186}

for pid in table2:
    f = next(raw.glob(f"Hurst_{pid}.txt"), None) or next(raw.glob(f"Hurst_{pid.lower()}.txt"))
    a = np.loadtxt(f)
    col = 0 if a.shape[1] == 2 else 2
    print(f"{pid:4s} SD(ddof=1)={a[:, col].std(ddof=1):.4f}  SD(ddof=0)={a[:, col].std(ddof=0):.4f}  table2={table2_sd[pid]}")

H3   SD(ddof=1)=0.0041  SD(ddof=0)=0.0041  table2=0.0041
H4   SD(ddof=1)=0.0115  SD(ddof=0)=0.0115  table2=0.0115
H5   SD(ddof=1)=0.0159  SD(ddof=0)=0.0159  table2=0.0159
H6   SD(ddof=1)=0.0076  SD(ddof=0)=0.0076  table2=0.0076
H7   SD(ddof=1)=0.0038  SD(ddof=0)=0.0038  table2=0.0038
H12  SD(ddof=1)=0.0061  SD(ddof=0)=0.0061  table2=0.0061
P1   SD(ddof=1)=0.0069  SD(ddof=0)=0.0069  table2=0.0069
P2   SD(ddof=1)=0.0094  SD(ddof=0)=0.0094  table2=0.0094
P8   SD(ddof=1)=0.0079  SD(ddof=0)=0.0079  table2=0.0079
P9   SD(ddof=1)=0.0084  SD(ddof=0)=0.0084  table2=0.0084
P10  SD(ddof=1)=0.0262  SD(ddof=0)=0.0262  table2=0.0262
P11  SD(ddof=1)=0.0186  SD(ddof=0)=0.0186  table2=0.0186


Checking which rows of the EEG files correspong to F3 by calculating the mean and sd of all the columns and seeing which column matches the sd and mean given in T1 of the original study for a given patient. We see its row 2 (3rd row) that contain the EEG numbers. 

In [8]:
from scipy.stats import skew, kurtosis

healthy = ["H3", "H4", "H5", "H6", "H7", "H12"]
ptsd = ["P1", "P2", "P8", "P9", "P10", "P11"]
eeg = {pid: np.loadtxt(raw / f"{pid}.txt") for pid in healthy + ptsd}

# Corrected Table 1: mean, SD, skewness, kurtosis
table1 = {"Healthy": np.array([29.2511, 15.7567, -0.4091, 4.8623]),
          "PTSD":    np.array([32.4275,  8.8147,  0.2089, 2.5751])}
groups = {"Healthy": healthy, "PTSD": ptsd}

def group_stats(row, ids):
    """Pool one row across a group's participants and compute the Table 1 statistics."""
    x = np.concatenate([eeg[pid][row] for pid in ids])
    return np.array([x.mean(), x.std(ddof=1), skew(x), kurtosis(x, fisher=False)])

# For each row, the largest difference from Table 1 across both groups and all four statistics
worst_diff = [max(np.abs(group_stats(r, ids) - table1[g]).max() for g, ids in groups.items())
              for r in range(32)]

best = int(np.argmin(worst_diff))
print(f"Best-matching row: {best}  (largest difference {worst_diff[best]:.4f})")
print(f"Next closest row:  {int(np.argsort(worst_diff)[1])}  (largest difference {sorted(worst_diff)[1]:.4f})\n")

for g, ids in groups.items():
    print(f"{g:8s} computed: {np.round(group_stats(best, ids), 4)}")
    print(f"{g:8s} Table 1:  {table1[g]}")

Best-matching row: 2  (largest difference 0.0002)
Next closest row:  23  (largest difference 2.9290)

Healthy  computed: [29.2511 15.7565 -0.4091  4.8624]
Healthy  Table 1:  [29.2511 15.7567 -0.4091  4.8623]
PTSD     computed: [32.4275  8.8147  0.2089  2.5752]
PTSD     Table 1:  [32.4275  8.8147  0.2089  2.5751]


checking participants with low variance channels, P11 (P6 on the box and whisker plot) has a near flat variance for 4 channels: TP9, TP10, FC1, FC2

In [9]:
sd = eeg["P11"].std(axis=1)
for row in np.argsort(sd)[:5]:
    print(f"row {row:2d}  SD = {sd[row]:.3f}")

row 28  SD = 0.319
row 29  SD = 0.485
row 20  SD = 0.539
row 21  SD = 0.867
row 16  SD = 3.196


In [10]:
for pid, data in eeg.items():
    sd = data.std(axis=1)
    flat = np.where(sd < 1)[0]
    if len(flat) > 0:
        print(pid, "near-flat rows:", flat, "SDs:", sd[flat].round(3))

P11 near-flat rows: [20 21 28 29] SDs: [0.539 0.867 0.319 0.485]


In [12]:

hurst = {}
for pid in healthy + ptsd:
    f = next(raw.glob(f"Hurst_{pid}.txt"), None) or next(raw.glob(f"Hurst_{pid.lower()}.txt"))
    hurst[pid] = np.loadtxt(f)

for pid, h in hurst.items():
    if h.shape[1] != 32:
        print(f"{pid:4s} skipped (only {h.shape[1]} columns)")
        continue
    means = h.mean(axis=0)
    eeg_means = means[:31]
    print(f"{pid:4s} col 31 mean = {means[31]:.4f}   "
          f"EEG cols: mean {eeg_means.mean():.4f}, range {eeg_means.min():.4f}–{eeg_means.max():.4f}")

H3   col 31 mean = 0.6271   EEG cols: mean 0.9135, range 0.8385–0.9752
H4   col 31 mean = 0.6053   EEG cols: mean 0.9586, range 0.9107–0.9811
H5   col 31 mean = 0.5937   EEG cols: mean 0.8940, range 0.8282–0.9536
H6   col 31 mean = 0.7883   EEG cols: mean 0.9468, range 0.8760–0.9883
H7   col 31 mean = 0.6559   EEG cols: mean 0.9326, range 0.9102–0.9640
H12  skipped (only 2 columns)
P1   col 31 mean = 0.6359   EEG cols: mean 0.9302, range 0.8955–0.9577
P2   col 31 mean = 0.6787   EEG cols: mean 0.9134, range 0.8749–0.9584
P8   col 31 mean = 0.5734   EEG cols: mean 0.9216, range 0.8759–0.9500
P9   col 31 mean = 0.6285   EEG cols: mean 0.9275, range 0.8769–0.9705
P10  col 31 mean = 0.7151   EEG cols: mean 0.9554, range 0.9322–0.9780
P11  skipped (only 2 columns)
